# 02 · Baseline · «¿Hoja sana o enferma, y cuándo no fiarnos del modelo? (Papas)» · Sprint 14

**Base de datos:** `PlantVillage-Dataset` (`https://github.com/spMohanty/PlantVillage-Dataset`), clases de papa (`sana` = `0` vs. `enferma` = `1`).

**Qué hacemos en este cuaderno (`02_baseline.ipynb`):**
1. Cargamos los datos de papa con `SEMILLA = 42` y definimos `OBJETIVO = "enferma"` (`X` = variables visuales de la hoja, `y` = etiqueta `enferma`).
2. Repartimos los datos de forma estratificada en entrenamiento (`_ent`), validación (`_val`) y test (`_test`, guardado sin tocar hasta tener candidato).
3. Construimos el **baseline** (`DummyClassifier(strategy="most_frequent")`) y calculamos su métrica (`F1_macro` y exactitud), cerrando con la frase obligatoria: **«Nuestro baseline tiene [métrica] de [valor]»**.

## 1. Cargar los datos y fijar semilla (`SEMILLA = 42`)

**Explicación en sencillo:** cargamos las imágenes de papa de `PlantVillage-Dataset` (con respaldo local automático si no hay red en Colab) y extraemos las variables visuales numéricas para repartir los datos y medir el baseline.

In [ ]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import f1_score, accuracy_score

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

CARPETA_DATOS = "datos_plantvillage_papas"
os.makedirs(CARPETA_DATOS, exist_ok=True)

def obtener_rutas_papas(destino):
    encontradas = []
    for raiz, _, archivos in os.walk(destino):
        for a in archivos:
            if a.lower().endswith((".jpg", ".jpeg", ".png")):
                encontradas.append(os.path.join(raiz, a))
    if len(encontradas) >= 30:
        return encontradas
    urls_zip = [
        "https://github.com/spMohanty/PlantVillage-Dataset/archive/refs/heads/master.zip",
        "https://codeload.github.com/spMohanty/PlantVillage-Dataset/zip/refs/heads/master"
    ]
    zip_path = os.path.join(destino, "plantvillage_master.zip")
    for url in urls_zip:
        try:
            urllib.request.urlretrieve(url, zip_path)
            with zipfile.ZipFile(zip_path, "r") as zf:
                miembros = [m for m in zf.namelist() if "Potato" in m and m.lower().endswith((".jpg", ".jpeg", ".png"))]
                sanas = [m for m in miembros if "healthy" in m.lower()][:90]
                enfermas = [m for m in miembros if "healthy" not in m.lower()][:180]
                for m in sanas + enfermas:
                    zf.extract(m, destino)
            extraidas = []
            for raiz, _, archivos in os.walk(destino):
                if "Potato" in raiz:
                    for a in archivos:
                        if a.lower().endswith((".jpg", ".jpeg", ".png")):
                            extraidas.append(os.path.join(raiz, a))
            if len(extraidas) >= 20:
                return extraidas
        except Exception:
            pass
    # Respaldo local automático
    rutas_loc = []
    for clase in ["Potato___healthy", "Potato___Early_blight", "Potato___Late_blight"]:
        c_dir = os.path.join(destino, clase)
        os.makedirs(c_dir, exist_ok=True)
        for i in range(80):
            img = Image.new("RGB", (128, 128), (232, 236, 230))
            draw = ImageDraw.Draw(img)
            col_hoja = (60, 140, 50) if "healthy" in clase else (85, 125, 48)
            draw.ellipse([24, 16, 104, 112], fill=col_hoja)
            if "blight" in clase:
                for _ in range(6):
                    cx, cy = int(rng.integers(35, 93)), int(rng.integers(28, 100))
                    draw.ellipse([cx-7, cy-7, cx+7, cy+7], fill=(80, 48, 30))
            arr = np.clip(np.asarray(img, dtype=np.float32) + rng.normal(0, 6.0, (128, 128, 3)), 0, 255).astype(np.uint8)
            p = os.path.join(c_dir, f"papa_{i:03d}.png")
            Image.fromarray(arr).save(p)
            rutas_loc.append(p)
    return rutas_loc

rutas_papas = obtener_rutas_papas(CARPETA_DATOS)
filas = []
for r_img in sorted(rutas_papas):
    subclase = next((c for c in r_img.split(os.sep) if "Potato" in c), "Potato___unknown")
    objetivo_enferma = 0 if "healthy" in subclase.lower() else 1
    with Image.open(r_img) as im:
        arr = np.asarray(im.convert("RGB").resize((64, 64)), dtype=np.float32) / 255.0
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    filas.append({
        "r_media": float(r.mean()),
        "g_media": float(g.mean()),
        "b_media": float(b.mean()),
        "prop_verde": float(((g > r + 0.04) & (g > b + 0.04)).mean()),
        "prop_mancha_oscura": float(((r < 0.35) & (g < 0.35) & (b < 0.35)).mean()),
        "enferma": objetivo_enferma
    })

df = pd.DataFrame(filas).sample(frac=1.0, random_state=SEMILLA).reset_index(drop=True)
print("Shape df:", df.shape)
df.info()

Shape df: (270, 6)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 270 entries, 0 to 269
Data columns (total 6 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   r_media             270 non-null    float64
 1   g_media             270 non-null    float64
 2   b_media             270 non-null    float64
 3   prop_verde          270 non-null    float64
 4   prop_mancha_oscura  270 non-null    float64
 5   enferma             270 non-null    int64  
dtypes: float64(5), int64(1)
memory usage: 12.8 KB


## 2. Repartir los datos en entrenamiento, validación y test

**Explicación en sencillo:** separamos `X` (las variables visuales de la hoja) e `y` (la columna `enferma`: `0` = sana, `1` = enferma). Primero apartamos el `20 %` para **test** (que no se toca hasta tener un candidato) y luego dividimos el resto en **entrenamiento** (`X_ent`, `y_ent`) y **validación** (`X_val`, `y_val`) manteniendo proporciones con `stratify` y `random_state=SEMILLA`.

In [ ]:
OBJETIVO = "enferma"
X = df.drop(columns=[OBJETIVO])
y = df[OBJETIVO]

X_resto, X_test, y_resto, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEMILLA
)
X_ent, X_val, y_ent, y_val = train_test_split(
    X_resto, y_resto, test_size=0.2, stratify=y_resto, random_state=SEMILLA
)
print("Tamaños (ent, val, test):", len(X_ent), len(X_val), len(X_test))

Tamaños (ent, val, test): 172 44 54


## 3. Construir el baseline y calcular su métrica

**Explicación en sencillo:** el `baseline` (`DummyClassifier(strategy="most_frequent")`) predice siempre la clase más frecuente en el entrenamiento. Es el suelo más sencillo que deberemos superar en los siguientes sprints; si el F1 de una clase sale `0.0` (o vemos un aviso *warning* de división a cero), significa que el baseline nunca predice la clase minoritaria.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent", random_state=SEMILLA).fit(X_ent, y_ent)
pred = baseline.predict(X_val)
acc_val = round(float(accuracy_score(y_val, pred)), 3)
f1_macro_val = round(float(f1_score(y_val, pred, average="macro", zero_division=0)), 3)
f1_bin_val = round(float(f1_score(y_val, pred, zero_division=0)), 3)

print(f"exactitud {acc_val} · F1_macro {f1_macro_val} · F1_binario {f1_bin_val}")
print(f"Nuestro baseline tiene F1_macro de {f1_macro_val} (y exactitud de {acc_val}).")

exactitud 0.659 · F1_macro 0.397 · F1_binario 0.795
Nuestro baseline tiene F1_macro de 0.397 (y exactitud de 0.659).
